# 무료 Colab에서 SSPNet 실행과 작은 수영자 평가

태인님을 위한 실행 노트북입니다. **무료 GPU 런타임만 사용**하며 결제·유료 서비스·상시 연결 기능을 사용하지 않습니다.

1. Colab에서 **파일 → 노트북 업로드**로 이 파일을 여세요.
2. **런타임 → 런타임 유형 변경 → GPU**를 선택하세요. 무료 GPU가 배정되지 않으면 여기서 멈추고 나중에 다시 시도하세요.
3. 1~7단계를 위에서부터 실행하면 환경 확인 → 공개 가중치 로드 → 예제 이미지 한 장 추론 → 결과 다운로드까지 진행합니다.
4. 실제 수영자 이미지는 6단계에서 업로드할 수 있습니다. 8단계 이후는 데이터셋 평가이며, 기본 범위는 **train·valid·test 전체**입니다.

이번 개정본은 AnnotationPiriapolis v4의 전체 635장과 `s=√(박스 가로×박스 세로)` 기준의 `2≤s<20`, `2≤s<8` 구간을 집계합니다. 다른 모델과의 비교표를 만들지 않고 이번 실행 결과의 문서용 요약을 저장합니다. 정답 개수 대조로 확인한 범위와 크기 구분을 사용하며, 외부 테스트의 미공개 임계값·IoU·입력 해상도까지 재현했다는 의미는 아닙니다.

**검증 범위:** 배포 전 Python 문법과 정답 변환·매칭·크기별 평가·분할 병합을 로컬에서 검사했습니다. 이 파일에 Colab GPU 실행 결과를 미리 넣지 않았습니다. 실제 CUDA·가중치·추론 성공은 아래 셀의 출력으로 확인하세요. 무료 GPU와 외부 다운로드의 가용성은 보장되지 않습니다.

**저장 위치:** 설치·원본 데이터는 Colab의 `/content`에 저장합니다. 이 노트북만 업로드하면 되며, 별도 `.py` 파일은 필요 없습니다. 런타임이 삭제되면 `/content`도 없어집니다. 세션을 넘어 이어서 평가하려면 2단계에서 Google Drive 저장을 켜고, 새 세션에서 데이터 ZIP을 다시 준비하세요.

**실험 범위:** 추가 학습 없이 TinyPerson 사전학습 SSPNet을 다른 해변 이미지에 적용합니다. 원 논문의 멀티스케일 평가 재현과 구분하세요. 이 노트북은 기본적으로 한 장씩·단일 해상도로 처리합니다.


## 1. GPU와 저장공간 확인

이 셀과 설치 셀은 무료 Colab의 Linux GPU 런타임에서 실행합니다. TPU와 CPU 런타임은 사용하지 않습니다. 여기서 표시하는 디스크·GPU는 태인님 노트북 사양이 아니라 Colab에 배정된 자원입니다.


In [ ]:
import os, sys, json, shutil, subprocess, platform, zipfile
from pathlib import Path
from IPython.display import display, Image as DisplayImage

WORK = Path("/content/sspnet_free")
if platform.system() != "Linux" or not Path("/content").exists():
    raise RuntimeError("이 노트북을 Google Colab에서 열어 실행하세요.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("무료 GPU 런타임이 필요합니다. 런타임 유형을 GPU로 변경하세요.")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=True)
disk = shutil.disk_usage("/content")
print("Colab 임시 디스크 여유: %.1f GiB" % (disk.free / 1024**3))
print("Colab 기본 Python:", sys.version.split()[0])
WORK.mkdir(exist_ok=True)


## 2. 결과 보관 위치 선택

처음 한 장만 확인할 때는 기본값 `False`로 실행해도 됩니다. 두 데이터셋 평가를 이어갈 때는 **`USE_GOOGLE_DRIVE = True`를 권합니다.** 인증 화면은 태인님이 직접 완료하세요.

Drive를 켜면 가중치와 이미지별 완료 기록·평가 결과를 `MyDrive/SSPNet_free_colab`에 보관합니다. 실제 이미지 처리와 설치는 `/content`에서 수행합니다. Drive의 남은 용량은 별도로 확인하세요. 무료 Colab 사용과 Google Drive의 저장공간은 별개입니다.


In [ ]:
USE_GOOGLE_DRIVE = False #@param {type:"boolean"}
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORE = Path("/content/drive/MyDrive/SSPNet_free_colab")
else:
    STORE = WORK / "saved"
STORE.mkdir(parents=True, exist_ok=True)
print("가중치·결과 저장:", STORE)
if not USE_GOOGLE_DRIVE:
    print("현재는 임시 저장입니다. 세션 종료 전에 마지막 결과 ZIP을 다운로드하세요.")


## 3. 실행 코드 준비

이 셀은 노트북에 포함된 보조 코드를 Colab 임시 폴더에 씁니다. 외부의 별도 보조 파일은 필요 없습니다.


In [ ]:
#@title 내장 실행 코드 준비
HELPER_SOURCES = {'setup_environment.py': '"""무료 Colab의 기본 Python과 분리하여 SSPNet 실행 환경을 만든다."""\nimport json\nimport os\nimport platform\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nROOT = Path("/content/sspnet_free")\nREPO = ROOT / "SSPNet"\nVENV = ROOT / "venv"\nPYTHON = VENV / "bin/python"\nCOMMIT = "f2d0f42273945577a5c6497fd2b384cb545f8f66"\nSTAMP = "python38-torch110-cu111-mmcv1318-v1"\n\n\ndef command(*args, env=None):\n    subprocess.run([str(a) for a in args], check=True, env=env)\n\n\ndef setup():\n    if platform.system() != "Linux" or platform.machine() != "x86_64" or not Path("/content").is_dir():\n        raise RuntimeError("이 설치 셀은 Google Colab Linux 런타임에서 실행하세요.")\n    if not shutil.which("nvidia-smi"):\n        raise RuntimeError("무료 Colab에서 런타임 유형 변경 → GPU를 선택한 뒤 다시 실행하세요.")\n    command("nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv")\n    free = shutil.disk_usage("/content").free / 1024**3\n    print("Colab 임시 디스크 여유: %.1f GiB" % free, flush=True)\n    if free < 8:\n        raise RuntimeError("환경 설치를 시작하려면 임시 디스크에 최소 8 GiB 여유가 필요합니다.")\n    ROOT.mkdir(exist_ok=True)\n    if not PYTHON.is_file():\n        command(sys.executable,"-m","pip","install","--quiet","--no-cache-dir","uv==0.9.7")\n        command("uv","python","install","3.8.20")\n        command("uv","venv","--python","3.8.20","--seed",VENV)\n    env = os.environ.copy()\n    env.pop("PYTHONPATH",None)\n    env.pop("PYTHONHOME",None)\n    env["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"\n    env["PYTHONNOUSERSITE"] = "1"\n    marker = ROOT / "environment_ready.json"\n    ready = json.loads(marker.read_text()).get("stamp") == STAMP if marker.exists() else False\n    if not ready:\n        command(PYTHON,"-m","pip","install","--no-cache-dir","pip==24.3.1",\n                "setuptools==59.5.0","wheel==0.45.1",env=env)\n        command(PYTHON,"-m","pip","install","--no-cache-dir",\n                "torch==1.10.0+cu111","torchvision==0.11.1+cu111",\n                "--find-links","https://download.pytorch.org/whl/cu111/torch_stable.html",env=env)\n        command(PYTHON,"-m","pip","install","--no-cache-dir",\n                "numpy==1.21.6","scipy==1.7.3","matplotlib==3.5.3","Pillow==9.5.0",\n                "opencv-python-headless==4.5.5.64","pycocotools==2.0.7",\n                "addict==2.4.0","yapf==0.31.0","PyYAML==6.0.1","packaging==23.2",\n                "terminaltables==3.1.10","six==1.16.0","tqdm==4.66.5","gdown==5.2.0",env=env)\n        # 사전 빌드된 연산자만 설치한다. 무료 GPU 시간으로 CUDA 소스 빌드를 하지 않는다.\n        command(PYTHON,"-m","pip","install","--no-cache-dir","--no-deps",\n                "https://download.openmmlab.com/mmcv/dist/cu111/torch1.10.0/"\n                "mmcv_full-1.3.18-cp38-cp38-manylinux1_x86_64.whl",env=env)\n    if not REPO.exists():\n        command("git","init",REPO)\n        command("git","-C",REPO,"remote","add","origin","https://github.com/MingboHong/SSPNet.git")\n        command("git","-C",REPO,"fetch","--depth=1","origin",COMMIT)\n        command("git","-C",REPO,"checkout","--detach","FETCH_HEAD")\n    actual = subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()\n    if actual != COMMIT:\n        raise RuntimeError("기존 SSPNet 코드 버전이 다릅니다. 새 무료 런타임에서 설치 셀부터 실행하세요.")\n    env["PYTHONPATH"] = str(REPO)\n    check = \'\'\'\nimport json, sys, torch, torchvision, mmcv, mmdet\nfrom mmcv.ops import nms, roi_align\nassert torch.__version__ == \'1.10.0+cu111\'\nassert mmcv.__version__ == \'1.3.18\'\nassert torch.cuda.is_available(), \'GPU를 사용할 수 없습니다.\'\na = torch.randn(32,32,device=\'cuda\'); (a @ a).sum().item()\nb = torch.tensor([[0.,0.,10.,10.],[1.,1.,9.,9.]],device=\'cuda\')\ns = torch.tensor([.9,.8],device=\'cuda\')\nnms(b,s,.5)\nroi_align(torch.ones(1,1,16,16,device=\'cuda\'),torch.tensor([[0.,1.,1.,8.,8.]],device=\'cuda\'),(3,3))\ntorch.cuda.synchronize()\nprint(json.dumps(dict(python=sys.version,torch=torch.__version__,torchvision=torchvision.__version__,\n    cuda=torch.version.cuda,mmcv=mmcv.__version__,mmdet=mmdet.__version__,\n    gpu=torch.cuda.get_device_name(0),capability=torch.cuda.get_device_capability(0)),ensure_ascii=False,indent=2))\n\'\'\'\n    command(PYTHON,"-c",check,env=env)\n    marker.write_text(json.dumps(dict(stamp=STAMP,source_commit=actual)),encoding="utf-8")\n    freeze = subprocess.check_output([str(PYTHON),"-m","pip","freeze"],env=env,text=True)\n    (ROOT/"installed_packages.txt").write_text(freeze,encoding="utf-8")\n    print("환경 확인 완료: CUDA 행렬 연산, NMS, RoIAlign, SSPNet 패키지 가져오기 성공",flush=True)\n\n\nif __name__ == "__main__":\n    setup()\n', 'metrics.py': '"""원본 이미지 좌표에서 수영자 탐지와 크기별 재현율을 평가한다."""\nimport json\nimport math\nimport re\nimport os\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\n\ndef canonical(value):\n    return " ".join(re.sub(r"[_-]+", " ", value.lower()).split())\n\n\ndef image_path(annotation_path, name):\n    base = Path(annotation_path).resolve().parent\n    path = (base / name).resolve()\n    if base != path and base not in path.parents:\n        raise ValueError("이미지 경로가 데이터셋 폴더 밖을 가리킵니다: " + name)\n    if not path.is_file():\n        raise FileNotFoundError("정답 파일과 같은 폴더 아래에 이미지가 필요합니다: " + str(path))\n    return path\n\n\ndef size_group(box):\n    d = max(float(box[2]), float(box[3]))\n    return "0<d<=8" if d <= 8 else "8<d<=12" if d <= 12 else "12<d<=20" if d <= 20 else "20<d"\n\n\ndef geometric_size(box):\n    return math.sqrt(float(box[2]) * float(box[3]))\n\n\ndef merge_splits(annotation_paths, output_path):\n    """분할별로 겹치는 ID와 파일명을 구분하고 이미지 복사 없이 합친다."""\n    output_path = Path(output_path).resolve()\n    base = output_path.parent\n    merged = dict(info={"description": "전체 분할 통합", "source_splits": []},\n                  licenses=[], images=[], annotations=[], categories=[])\n    categories = {}\n    seen_paths = set()\n    for split, source in annotation_paths:\n        source = Path(source).resolve()\n        if split in merged["info"]["source_splits"]:\n            raise ValueError("같은 분할을 두 번 합칠 수 없습니다: " + split)\n        raw = json.loads(source.read_text(encoding="utf-8"))\n        merged["info"]["source_splits"].append(split)\n        catmap = {}\n        for category in raw["categories"]:\n            name = canonical(category["name"])\n            if name not in categories:\n                categories[name] = len(categories) + 1\n                merged["categories"].append(dict(id=categories[name], name=category["name"]))\n            catmap[category["id"]] = categories[name]\n        images = {}\n        for im in sorted(raw["images"], key=lambda v: v["id"]):\n            if im["id"] in images:\n                raise ValueError("분할 안에서 이미지 ID가 중복됩니다: " + split)\n            path = image_path(source, im["file_name"])\n            if path in seen_paths:\n                raise ValueError("같은 이미지 경로가 중복됩니다: " + str(path))\n            seen_paths.add(path)\n            if base not in path.parents:\n                raise ValueError("통합 정답은 모든 이미지의 상위 폴더에 저장해야 합니다.")\n            new_id = len(merged["images"])\n            images[im["id"]] = new_id\n            merged["images"].append(dict(im, id=new_id, file_name=path.relative_to(base).as_posix(),\n                                         source_split=split, source_image_id=im["id"]))\n        seen_annotations = set()\n        for a in raw["annotations"]:\n            if a["id"] in seen_annotations:\n                raise ValueError("분할 안에서 정답 ID가 중복됩니다: " + split)\n            seen_annotations.add(a["id"])\n            if a["image_id"] not in images or a["category_id"] not in catmap:\n                raise ValueError("정답의 이미지 또는 종류 ID를 찾지 못했습니다: " + split)\n            merged["annotations"].append(dict(a, id=len(merged["annotations"])+1,\n                image_id=images[a["image_id"]], category_id=catmap[a["category_id"]],\n                source_split=split, source_annotation_id=a["id"]))\n    if not merged["images"]:\n        raise ValueError("합칠 이미지가 없습니다.")\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = output_path.with_suffix(".json.tmp")\n    temporary.write_text(json.dumps(merged, ensure_ascii=False, indent=2), encoding="utf-8")\n    os.replace(str(temporary), str(output_path))\n    return output_path\n\n\ndef load_dataset(annotation_path, target_names=("person water",), limit=0):\n    raw = json.loads(Path(annotation_path).read_text(encoding="utf-8"))\n    cats = raw.get("categories", [])\n    wanted = {canonical(s) for s in target_names}\n    selected = [c["id"] for c in cats if canonical(c["name"]) in wanted]\n    if not selected:\n        raise ValueError("물속 사람 라벨을 찾지 못했습니다. 실제 종류: " + str([c["name"] for c in cats]))\n    images = sorted(raw["images"], key=lambda im: im["id"])\n    if len({im["id"] for im in images}) != len(images):\n        raise ValueError("이미지 ID가 중복되어 있습니다.")\n    if limit < 0:\n        raise ValueError("이미지 제한은 0 이상이어야 합니다.")\n    if limit:\n        images = images[:limit]\n    if not images:\n        raise ValueError("선택한 분할에 이미지가 없습니다.")\n    imap = {im["id"]: im for im in images}\n    ann_ids = set()\n    annotations = []\n    for a in raw["annotations"]:\n        if a["id"] in ann_ids:\n            raise ValueError("정답 ID가 중복되어 있습니다.")\n        ann_ids.add(a["id"])\n        if a["image_id"] not in imap or a["category_id"] not in selected:\n            continue\n        x, y, w, h = map(float, a["bbox"])\n        im = imap[a["image_id"]]\n        if not all(math.isfinite(v) for v in (x, y, w, h)) or w <= 0 or h <= 0:\n            raise ValueError("유효하지 않은 정답 박스: " + str(a["id"]))\n        if x < -.5 or y < -.5 or x + w > im["width"] + .5 or y + h > im["height"] + .5:\n            raise ValueError("이미지 밖의 정답 박스입니다. 좌표 변환을 확인하세요: " + str(a["id"]))\n        annotations.append(dict(id=a["id"], image_id=a["image_id"], category_id=1,\n                                bbox=[x, y, w, h], area=w*h,\n                                iscrowd=int(a.get("iscrowd", 0)),\n                                ignore=bool(a.get("ignore", False)), segmentation=[]))\n    for im in images:\n        path = image_path(annotation_path, im["file_name"])\n        from PIL import Image\n        with Image.open(path) as pic:\n            if pic.size != (im["width"], im["height"]):\n                raise ValueError("이미지 크기와 정답 메타데이터가 다릅니다: " + str(path))\n    gt = dict(info={"description": "물속 사람 평가용 정답"}, licenses=[], images=images,\n              annotations=annotations, categories=[dict(id=1, name="person water")])\n    counts = Counter(size_group(a["bbox"]) for a in annotations if not a["ignore"] and not a["iscrowd"])\n    normal = [a for a in annotations if not a["ignore"] and not a["iscrowd"]]\n    split_images = Counter(im.get("source_split", "single") for im in images)\n    image_splits = {im["id"]: im.get("source_split", "single") for im in images}\n    split_targets = Counter(image_splits[a["image_id"]] for a in normal)\n    return gt, dict(image_count=len(images), available_image_count=len(raw["images"]), target_count=sum(counts.values()),\n                    geometric_2to20_count=sum(2 <= geometric_size(a["bbox"]) < 20 for a in normal),\n                    geometric_2to8_count=sum(2 <= geometric_size(a["bbox"]) < 8 for a in normal),\n                    split_counts={s: dict(images=n, water_targets=split_targets[s]) for s,n in split_images.items()},\n                    size_counts=dict(counts), available_classes=[c["name"] for c in cats],\n                    target_classes=[c["name"] for c in cats if c["id"] in selected])\n\n\ndef overlap(a, b, crowd=False):\n    x, y, w, h = a\n    u, v, s, t = b\n    inter = max(0., min(x+w, u+s)-max(x, u)) * max(0., min(y+h, v+t)-max(y, v))\n    area = w*h if crowd else w*h+s*t-inter\n    return inter / area if area > 0 else 0.\n\n\ndef match_predictions(gt, predictions, iou=.5):\n    normal, ignored = defaultdict(list), defaultdict(list)\n    for a in gt["annotations"]:\n        (ignored if a.get("ignore") or a.get("iscrowd") else normal)[a["image_id"]].append(a)\n    used = set()\n    valid_ids = {im["id"] for im in gt["images"]}\n    ranked = sorted((p for p in predictions if p["image_id"] in valid_ids), key=lambda p: -p["score"])\n    matches = []\n    for p in ranked:\n        candidates = [(overlap(p["bbox"], a["bbox"]), a) for a in normal[p["image_id"]] if a["id"] not in used]\n        score, best = max(candidates, key=lambda pair: pair[0]) if candidates else (0., None)\n        if best is not None and score >= iou:\n            used.add(best["id"])\n            matches.append((p, best, False))\n        elif any(overlap(p["bbox"], a["bbox"], bool(a.get("iscrowd"))) >= iou for a in ignored[p["image_id"]]):\n            matches.append((p, None, True))\n        else:\n            matches.append((p, None, False))\n    return matches\n\n\ndef evaluate(gt, predictions, score_threshold=.25, iou=.5):\n    matches = match_predictions(gt, predictions, iou)\n    counts = Counter(size_group(a["bbox"]) for a in gt["annotations"] if not a.get("ignore") and not a.get("iscrowd"))\n    positives = sum(counts.values())\n    hits, tp, fp = Counter(), 0, 0\n    recalls, precisions = [], []\n    total_tp = total_fp = 0\n    for p, a, ignored in matches:\n        if ignored:\n            continue\n        total_tp += a is not None\n        total_fp += a is None\n        recalls.append(total_tp / positives if positives else 0.)\n        precisions.append(total_tp / (total_tp + total_fp))\n        if p["score"] >= score_threshold:\n            tp += a is not None\n            fp += a is None\n            if a is not None:\n                hits[size_group(a["bbox"])] += 1\n    ap = sum(max((p for r, p in zip(recalls, precisions) if r >= q/100), default=0.) for q in range(101))/101 if positives else None\n    tiny_count = sum(counts[k] for k in ("0<d<=8", "8<d<=12", "12<d<=20"))\n    tiny_hits = sum(hits[k] for k in ("0<d<=8", "8<d<=12", "12<d<=20"))\n    def geometric_range(low, high):\n        selected = {a["id"] for a in gt["annotations"] if not a.get("ignore") and not a.get("iscrowd")\n                    and low <= geometric_size(a["bbox"]) < high}\n        found = sum(a is not None and a["id"] in selected for p,a,ignored in matches\n                    if not ignored and p["score"] >= score_threshold)\n        return dict(positives=len(selected), hits=found, recall=found/len(selected) if selected else None)\n    return dict(iou=iou, score_threshold=score_threshold, images=len(gt["images"]),\n                positives=positives, true_positives=tp, false_positives=fp,\n                precision=tp/(tp+fp) if tp+fp else None,\n                recall=tp/positives if positives else None,\n                false_positives_per_image=fp/len(gt["images"]),\n                AP50_saved_predictions_101point=ap if iou == .5 else None,\n                geometric_size_definition="s=sqrt(width*height), 배포 이미지 픽셀 기준",\n                geometric_2to20=geometric_range(2,20),\n                geometric_2to8=geometric_range(2,8),\n                geometric_size_recall={"2<=s<8": geometric_range(2,8),\n                                       "8<=s<12": geometric_range(8,12),\n                                       "12<=s<20": geometric_range(12,20)},\n                tiny_le20=dict(positives=tiny_count, hits=tiny_hits,\n                               recall=tiny_hits/tiny_count if tiny_count else None),\n                size_recall={k: dict(positives=counts[k], hits=hits[k],\n                                    recall=hits[k]/counts[k] if counts[k] else None)\n                             for k in ("0<d<=8", "8<d<=12", "12<d<=20", "20<d")})\n', 'engine.py': '"""SSPNet 추론, 분할 병합, 이미지별 재시작 캐시."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageDraw\nfrom metrics import evaluate, image_path, load_dataset\n\n\ndef dump(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    os.replace(str(tmp), str(path))\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with open(path, "rb") as f:\n        for block in iter(lambda: f.read(1024*1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef positions(length, size, overlap):\n    if size <= 0 or not 0 <= overlap < 1:\n        raise ValueError("분할 크기는 양수, 겹침은 0 이상 1 미만이어야 합니다.")\n    if length <= size:\n        return [0]\n    step = max(1, int(size*(1-overlap)))\n    values = list(range(0, length-size+1, step))\n    if values[-1] != length-size:\n        values.append(length-size)\n    return values\n\n\ndef merge_nms(boxes, threshold=.5):\n    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 5)\n    if not len(boxes):\n        return boxes\n    order = np.argsort(-boxes[:, 4], kind="stable")\n    keep = []\n    while len(order):\n        i = order[0]\n        keep.append(i)\n        rest = order[1:]\n        b = boxes[rest]\n        inter = np.maximum(0, np.minimum(boxes[i, 2:4], b[:, 2:4])-np.maximum(boxes[i, :2], b[:, :2])).prod(axis=1)\n        area = np.maximum(0, boxes[i, 2:4]-boxes[i, :2]).prod()\n        areas = np.maximum(0, b[:, 2:4]-b[:, :2]).prod(axis=1)\n        union = area+areas-inter\n        iou = inter/np.maximum(union, 1e-12)\n        order = rest[iou <= threshold]\n    return boxes[keep]\n\n\ndef make_model(settings):\n    import torch\n    from mmcv import Config\n    from mmdet.models import build_detector\n    if not torch.cuda.is_available():\n        raise RuntimeError("GPU가 없습니다. 무료 Colab에서 GPU 런타임을 선택하세요.")\n    torch.set_num_threads(2)\n    torch.manual_seed(0)\n    np.random.seed(0)\n    torch.backends.cudnn.benchmark = False\n    cfg = Config.fromfile(str(Path(settings["repo"])/"configs/_base_/models/faster_rcnn_r50_sspnet.py"))\n    cfg.model.pretrained = None\n    cfg.model.train_cfg = None\n    cfg.test_cfg.rcnn.score_thr = settings["save_min_score"]\n    cfg.test_cfg.rcnn.nms.min_score = settings["save_min_score"]\n    cfg.test_cfg.rcnn.max_per_img = settings["max_detections"]\n    cfg.data = dict(test=dict(pipeline=[\n        dict(type="LoadImageFromFile"),\n        dict(type="MultiScaleFlipAug", img_scale=(settings["long_edge"], settings["short_edge"]),\n             flip=False, transforms=[dict(type="Resize", keep_ratio=True), dict(type="RandomFlip"),\n             dict(type="Normalize", mean=[123.675,116.28,103.53], std=[58.395,57.12,57.375], to_rgb=True),\n             dict(type="Pad", size_divisor=32), dict(type="ImageToTensor", keys=["img"]),\n             dict(type="Collect", keys=["img"])])]))\n    model = build_detector(cfg.model, test_cfg=cfg.test_cfg)\n    checkpoint = torch.load(settings["checkpoint"], map_location="cpu")\n    state = checkpoint.get("state_dict", checkpoint)\n    if all(k.startswith("module.") for k in state):\n        state = {k[7:]: v for k,v in state.items()}\n    if not any("neck.CAM." in k for k in state):\n        raise ValueError("SSPNet 가중치가 아닙니다. Faster R-CNN SSPNet용 파일을 확인하세요.")\n    model.load_state_dict(state, strict=True)\n    model.CLASSES = ("person",)\n    model.cfg = cfg\n    model.cuda().eval()\n    return model, cfg\n\n\ndef predict(model, image, settings):\n    import torch\n    from mmdet.apis import inference_detector\n    rgb = np.asarray(Image.open(image).convert("RGB"))\n    h, w = rgb.shape[:2]\n    torch.cuda.reset_peak_memory_stats()\n    torch.cuda.synchronize()\n    started = time.perf_counter()\n    patches = 0\n    boxes = []\n    if settings["mode"] == "full":\n        regions = [(0, 0, w, h)]\n    else:\n        size = settings["tile_size"]\n        regions = [(x,y,min(x+size,w),min(y+size,h))\n                   for y in positions(h,size,settings["tile_overlap"])\n                   for x in positions(w,size,settings["tile_overlap"])]\n    for x,y,r,b in regions:\n        # MMDetection의 배열 입력은 BGR 순서를 사용한다.\n        patch = np.ascontiguousarray(rgb[y:b,x:r,::-1])\n        result = inference_detector(model, patch)\n        if not isinstance(result, list) or len(result) != 1:\n            raise ValueError("단일 person 클래스 SSPNet 출력이 아닙니다.")\n        out = np.asarray(result[0], dtype=np.float32).reshape(-1,5).copy()\n        out[:,[0,2]] += x\n        out[:,[1,3]] += y\n        boxes.extend(out.tolist())\n        patches += 1\n    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1,5)\n    if len(boxes):\n        boxes[:,[0,2]] = boxes[:,[0,2]].clip(0,w)\n        boxes[:,[1,3]] = boxes[:,[1,3]].clip(0,h)\n        valid = np.isfinite(boxes).all(axis=1) & (boxes[:,2]>boxes[:,0]) & (boxes[:,3]>boxes[:,1])\n        boxes = boxes[valid]\n    if settings["mode"] == "tiles":\n        boxes = merge_nms(boxes,settings["merge_iou"])\n    if len(boxes):\n        boxes = boxes[np.argsort(-boxes[:,4],kind="stable")][:settings["max_detections"]]\n    torch.cuda.synchronize()\n    return dict(boxes=boxes.tolist(), seconds=time.perf_counter()-started, patches=patches,\n                peak_vram_mib=torch.cuda.max_memory_allocated()/1024**2)\n\n\ndef draw(image, record, path, threshold):\n    picture = Image.open(image).convert("RGB")\n    pen = ImageDraw.Draw(picture)\n    for x,y,r,b,s in record["boxes"]:\n        if s >= threshold:\n            pen.rectangle((x,y,r,b),outline="red",width=2)\n            pen.text((x,max(0,y-12)),"person %.2f" % s,fill="red")\n    picture.save(path)\n\n\ndef coco_predictions(records):\n    return [dict(image_id=im_id,category_id=1,bbox=[x,y,r-x,b-y],score=s)\n            for im_id,rec in records.items() for x,y,r,b,s in rec["boxes"]]\n\n\ndef report_text(metrics, settings, summary):\n    def percent(value):\n        return "계산 불가" if value is None else "%.1f%%" % (100*value)\n    small = metrics["geometric_2to20"]\n    smallest = metrics["geometric_2to8"]\n    method = "이미지 전체를 한 번에 검사" if settings["mode"] == "full" else \\\n        "%d픽셀 패치로 분할 검사(겹침 %.0f%%)" % (settings["tile_size"],settings["tile_overlap"]*100)\n    lines = ["SSPNet 수영자 탐지 평가", "",\n             "평가 데이터: %s / 분할: %s / 이미지 %d장" %\n             (settings.get("dataset_name","사용자 데이터"), settings.get("split","single"), metrics["images"]),\n             "평가 방식: " + method,\n             "공개 TinyPerson 학습 가중치를 사용했으며 이번 데이터로 추가 학습하지 않았다.",\n             "전체 물속 사람 정답 %d개 중 %d개를 찾아 재현율 %s, 정밀도 %s였다." %\n             (metrics["positives"],metrics["true_positives"],percent(metrics["recall"]),percent(metrics["precision"])),\n             "사람 크기는 배포 이미지에서 s=√(박스 가로×박스 세로)로 정의했다.",\n             "2≤s<20픽셀 물속 사람 %d개 중 %d개를 찾아 재현율 %s였다." %\n             (small["positives"],small["hits"],percent(small["recall"])),\n             "가장 작은 2≤s<8픽셀 구간에서는 %d개 중 %d개를 찾아 재현율 %s였다." %\n             (smallest["positives"],smallest["hits"],percent(smallest["recall"])),\n             "오탐은 %d개(이미지당 %.2f개)였고, 점수 임계값 %.3f와 IoU %.2f를 적용했다." %\n             (metrics["false_positives"],metrics["false_positives_per_image"],metrics["score_threshold"],metrics["iou"])]\n    if settings.get("split") == "all":\n        lines.append("train·valid·test와 배포된 증강 이미지를 포함한 전체 분할 평가이며, 독립 테스트 분할 성능과 구분한다.")\n    if metrics["partial"]:\n        lines.append("현재 결과는 일부 이미지만 처리한 표본 평가다.")\n    lines += ["", "분할별 구성: " + json.dumps(summary["split_counts"],ensure_ascii=False),\n              "입력 해상도 상한: %d×%d, 종횡비 유지 / 최대 탐지 수: %d" %\n              (settings["long_edge"],settings["short_edge"],settings["max_detections"]),\n              "검출 결과는 전체 물속 사람 정답과 일대일로 매칭한 뒤 크기 구간별로 집계했다.",\n              "이 파일은 이번 SSPNet 실행 결과만 정리하며 다른 모델과의 비교는 포함하지 않는다."]\n    return "\\n".join(lines) + "\\n"\n\n\ndef run(args):\n    import torch\n    settings = json.loads(Path(args.settings).read_text(encoding="utf-8"))\n    if settings["mode"] not in ("full","tiles"):\n        raise ValueError("추론 방식은 full 또는 tiles여야 합니다.")\n    if not 0 < settings["save_min_score"] <= settings["report_score"] <= 1:\n        raise ValueError("저장 점수와 보고 점수의 범위를 확인하세요.")\n    if not 0 < settings.get("match_iou",.5) <= 1:\n        raise ValueError("정답 매칭 IoU는 0 초과 1 이하여야 합니다.")\n    if settings["mode"] == "tiles":\n        positions(1,settings["tile_size"],settings["tile_overlap"])\n    out = Path(args.out)\n    out.mkdir(parents=True,exist_ok=True)\n    settings["checkpoint_sha256"] = sha256(settings["checkpoint"])\n    settings["source_commit"] = subprocess.check_output(["git","-C",settings["repo"],"rev-parse","HEAD"],text=True).strip()\n    settings["engine_sha256"] = sha256(__file__)\n    settings["metrics_sha256"] = sha256(Path(__file__).with_name("metrics.py"))\n    if args.image:\n        model,cfg = make_model(settings)\n        cfg.dump(str(out/"resolved_config.py"))\n        # 첫 호출의 초기화 비용은 별도 기록하고 본 측정에서 제외한다.\n        warmup = predict(model,args.image,dict(settings,mode="full"))\n        rec = predict(model,args.image,settings)\n        rec.update(image_sha256=sha256(args.image), image_file_name=Path(args.image).name)\n        dump(out/"predictions.json",rec)\n        dump(out/"settings.json",dict(settings,warmup_seconds=warmup["seconds"],gpu=torch.cuda.get_device_name(0)))\n        draw(args.image,rec,out/"prediction.jpg",settings["report_score"])\n        print("이미지 추론 완료: 박스 %d개, 표시 임계값 %.2f, %.3f초" %\n              (len(rec["boxes"]),settings["report_score"],rec["seconds"]))\n        print("출력:",out)\n        return\n    gt,summary = load_dataset(args.annotations,settings["target_names"],args.limit)\n    if not summary["target_count"]:\n        raise ValueError("선택한 이미지에 평가할 물속 사람 정답이 없습니다.")\n    dump(out/"dataset_summary.json",summary)\n    settings["annotations_sha256"] = sha256(args.annotations)\n    identity = dict(settings)\n    # 결과 표시 임계값을 바꿔도 저점수 예측을 재사용할 수 있다.\n    identity.pop("report_score",None)\n    key = hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()[:16]\n    cache = out/("cache_"+key)\n    cache.mkdir(exist_ok=True)\n    dump(cache/"identity.json",identity)\n    dump(out/"run_status.json",dict(state="실행 중",cache_key=key,\n         image_ids=[im["id"] for im in gt["images"]]))\n    records = {}\n    pending = []\n    for im in gt["images"]:\n        path = image_path(args.annotations,im["file_name"])\n        image_hash = sha256(path)\n        saved = cache/(str(im["id"])+".json")\n        try:\n            rec = json.loads(saved.read_text(encoding="utf-8"))\n        except (FileNotFoundError,json.JSONDecodeError):\n            rec = None\n        if rec and rec.get("image_sha256") == image_hash:\n            records[im["id"]] = rec\n        else:\n            pending.append((im,path,image_hash,saved))\n    warmup_seconds = None\n    if pending:\n        model,cfg = make_model(settings)\n        cfg.dump(str(out/"resolved_config.py"))\n        warmup_seconds = predict(model,pending[0][1],dict(settings,mode="full"))["seconds"]\n        for im,path,image_hash,saved in pending:\n            rec = predict(model,path,settings)\n            rec.update(image_sha256=image_hash,file_name=im["file_name"])\n            dump(saved,rec)\n            records[im["id"]] = rec\n            print("완료 %d/%d: %s" % (len(records),len(gt["images"]),im["file_name"]),flush=True)\n            # 적은 수의 시각화만 저장하여 무료 저장공간을 아낀다.\n            if len(records) <= 5:\n                draw(path,rec,out/("preview_%s.jpg" % im["id"]),settings["report_score"])\n    predictions = coco_predictions(records)\n    dump(out/"ground_truth_water.json",gt)\n    dump(out/"predictions_coco.json",predictions)\n    metrics = evaluate(gt,predictions,settings["report_score"],settings.get("match_iou",.5))\n    metrics.update(partial=len(gt["images"]) < summary["available_image_count"],requested_limit=args.limit,cache_key=key,\n                   mean_inference_seconds=sum(r["seconds"] for r in records.values())/len(records),\n                   maximum_vram_mib=max(r["peak_vram_mib"] for r in records.values()),\n                   warmup_seconds=warmup_seconds)\n    metrics["by_split"] = {}\n    for split in summary["split_counts"]:\n        image_ids = {im["id"] for im in gt["images"] if im.get("source_split","single") == split}\n        split_gt = dict(gt, images=[im for im in gt["images"] if im["id"] in image_ids],\n                        annotations=[a for a in gt["annotations"] if a["image_id"] in image_ids])\n        metrics["by_split"][split] = evaluate(split_gt,predictions,settings["report_score"],settings.get("match_iou",.5))\n    dump(out/"metrics.json",metrics)\n    (out/"문서용_결과요약.txt").write_text(report_text(metrics,settings,summary),encoding="utf-8")\n    dump(out/"run_manifest.json",dict(settings,dataset=summary,\n         image_ids=[im["id"] for im in gt["images"]],gpu=torch.cuda.get_device_name(0)))\n    dump(out/"run_status.json",dict(state="완료",cache_key=key,\n         image_ids=[im["id"] for im in gt["images"]]))\n    with open(out/"metrics.csv","w",encoding="utf-8-sig",newline="") as f:\n        writer=csv.writer(f)\n        writer.writerow(["지표","값"])\n        for name,value in metrics.items():\n            if not isinstance(value,dict):\n                writer.writerow([name,value])\n        for name,value in metrics["size_recall"].items():\n            writer.writerow(["크기별 재현율 "+name,value["recall"]])\n        for name in ("geometric_2to20","geometric_2to8"):\n            for item,value in metrics[name].items():\n                writer.writerow([name+" "+item,value])\n        for name,value in metrics["geometric_size_recall"].items():\n            writer.writerow(["기하평균 크기별 재현율 "+name,value["recall"]])\n    print(json.dumps(metrics,ensure_ascii=False,indent=2))\n\n\nif __name__ == "__main__":\n    parser=argparse.ArgumentParser(description="SSPNet 단일 이미지 추론과 물속 사람 평가")\n    parser.add_argument("--settings",required=True)\n    parser.add_argument("--out",required=True)\n    group=parser.add_mutually_exclusive_group(required=True)\n    group.add_argument("--image")\n    group.add_argument("--annotations")\n    parser.add_argument("--limit",type=int,default=0)\n    run(parser.parse_args())\n'}
HELPERS = WORK / "helpers"
HELPERS.mkdir(exist_ok=True)
for name, source in HELPER_SOURCES.items():
    (HELPERS / name).write_text(source, encoding="utf-8")
print("보조 코드 준비 완료:", list(HELPER_SOURCES))


## 4. Python 3.8 환경 설치와 GPU 연산 검사

Colab 기본 Python·PyTorch는 바꾸지 않습니다. `uv`로 별도 Python 3.8.20 환경을 만들고 모든 SSPNet 명령을 그 환경의 Python으로 실행합니다.

| 구성 | 고정 버전 |
|---|---|
| Python | 3.8.20 |
| PyTorch / torchvision | 1.10.0+cu111 / 0.11.1+cu111 |
| MMCV | mmcv-full 1.3.18, 사전 빌드 wheel |
| NumPy | 1.21.6 |
| SSPNet 코드 | `f2d0f42273945577a5c6497fd2b384cb545f8f66` |

README의 Python 3.7·CUDA 10.2 대신, 배포된 Linux wheel이 있는 Python 3.8·CUDA 11.1 조합을 사용합니다. CUDA 툴킷 전체를 설치하거나 MMCV를 소스에서 컴파일하지 않습니다. 첫 설치는 네트워크에 따라 수 분 이상 걸리고 PyTorch 다운로드가 큽니다. 완료 전에 다음 셀을 실행하지 마세요.

설치 마지막에 **CUDA 행렬 연산·NMS·RoIAlign**이 성공해야 합니다. 실패하면 출력의 첫 오류를 확인하세요. 새 GPU에서 구형 CUDA 연산이 지원되지 않는 경우까지 자동으로 해결한다고 가정하지 않습니다.


In [ ]:
def checked(command, env=None):
    subprocess.run([str(x) for x in command], check=True, env=env)

checked([sys.executable, HELPERS / "setup_environment.py"])
PYTHON = WORK / "venv/bin/python"
REPO = WORK / "SSPNet"
ENV = os.environ.copy()
ENV.pop("PYTHONHOME", None)
ENV["PYTHONNOUSERSITE"] = "1"
ENV["PYTHONPATH"] = str(REPO) + os.pathsep + str(HELPERS)
ENV["MPLBACKEND"] = "Agg"
shutil.copy2(WORK / "installed_packages.txt", STORE / "installed_packages.txt")
print("SSPNet 전용 Python:", PYTHON)


## 5. Faster R-CNN SSPNet 가중치 준비

모델 저자가 README에 연결한 **Faster R-CNN SSPNet** 가중치를 받습니다. Fovea·일반 Faster R-CNN·YOLO 가중치를 섞어 쓰지 마세요. 7단계에서 모든 모델 파라미터를 엄격하게 대조하며, 누락·불일치를 무시하지 않습니다.

공유 다운로드가 제한되면 [저자의 다운로드 링크](https://drive.google.com/file/d/1IfPCt5xZqqBJ3sYVIuD5F9l29Jcy2Hn1/view)에서 직접 받은 `.pth`를 사용하세요. `WEIGHT_SOURCE`를 `upload`로 바꾸면 업로드할 수 있고, `path`는 기존 Colab/Drive 파일을 사용합니다. 캐시된 가중치가 있으면 다시 다운로드하지 않습니다.


In [ ]:
WEIGHT_SOURCE = "download" #@param ["download", "upload", "path"]
EXISTING_WEIGHT_PATH = "" #@param {type:"string"}
WEIGHT_STORE = STORE / "weights/faster_rcnn_sspnet.pth"
WEIGHT_STORE.parent.mkdir(exist_ok=True)
CHECKPOINT = WORK / "faster_rcnn_sspnet.pth"
if WEIGHT_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("가중치 파일 한 개만 선택하세요.")
    CHECKPOINT.write_bytes(next(iter(uploaded.values())))
elif WEIGHT_SOURCE == "path":
    source = Path(EXISTING_WEIGHT_PATH)
    if not source.is_file():
        raise FileNotFoundError("기존 가중치 경로를 확인하세요.")
    if source.resolve() != CHECKPOINT.resolve():
        shutil.copy2(source, CHECKPOINT)
elif WEIGHT_SOURCE == "download":
    if WEIGHT_STORE.is_file() and WEIGHT_STORE.stat().st_size > 10_000_000:
        if WEIGHT_STORE.resolve() != CHECKPOINT.resolve():
            shutil.copy2(WEIGHT_STORE, CHECKPOINT)
    else:
        download_code = (
            "import gdown; p=gdown.download(id='1IfPCt5xZqqBJ3sYVIuD5F9l29Jcy2Hn1', "
            "output=" + repr(str(CHECKPOINT)) + ", quiet=False); "
            "assert p, '가중치 다운로드에 실패했습니다.'"
        )
        checked([PYTHON, "-c", download_code], ENV)
else:
    raise ValueError("가중치 입력 방식을 확인하세요.")
if not CHECKPOINT.is_file() or CHECKPOINT.stat().st_size < 10_000_000:
    raise ValueError("가중치 파일이 너무 작습니다. 다운로드 오류 또는 HTML 파일인지 확인하세요.")
print("가중치 준비:", CHECKPOINT, "%.1f MiB" % (CHECKPOINT.stat().st_size / 1024**2))


## 6. 추론할 이미지 한 장 선택

기본 `demo`는 SSPNet 저장소의 예제 이미지로 **실행 경로 확인용**이며 수영자 성능 평가용이 아닙니다. 태인님의 수영자 이미지가 있으면 `upload`, 이미 Colab에 있다면 `path`를 선택하세요. 업로드 이미지가 작게 보이더라도 정답 박스의 크기는 확대 전 픽셀 기준으로 판단합니다.


In [ ]:
IMAGE_SOURCE = "demo" #@param ["demo", "upload", "path"]
EXISTING_IMAGE_PATH = "" #@param {type:"string"}
if IMAGE_SOURCE == "demo":
    IMAGE_PATH = REPO / "demo/demo.jpg"
elif IMAGE_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("이미지 한 장만 선택하세요.")
    name, content = next(iter(uploaded.items()))
    IMAGE_PATH = WORK / ("input" + Path(name).suffix.lower())
    IMAGE_PATH.write_bytes(content)
else:
    IMAGE_PATH = Path(EXISTING_IMAGE_PATH)
if not IMAGE_PATH.is_file():
    raise FileNotFoundError("이미지 경로를 확인하세요.")
display(DisplayImage(filename=str(IMAGE_PATH), width=900))


## 7. 이미지 한 장 추론과 결과 확인

`REPORT_SCORE`는 화면 표시와 재현율 보고용 임계값입니다. 낮은 점수의 박스도 `SAVE_MIN_SCORE` 이상이면 저장하여 나중에 정밀도·재현율을 비교할 수 있습니다. 출력 라벨 `person`은 모델의 예측 클래스이며, 수영 또는 익수 상태를 판단했다는 의미가 아닙니다.

`full`은 이미지 전체, `tiles`는 겹치는 패치로 분할하는 방식입니다. 먼저 기본 `full`로 동작을 확인하세요. 메모리 부족이면 해상도를 낮추거나 한 장씩 분할하세요. 해상도를 바꿨다면 별도 실험으로 기록해야 합니다.


In [ ]:
MODE = "full" #@param ["full", "tiles"]
LONG_EDGE = 1280 #@param {type:"integer"}
SHORT_EDGE = 1024 #@param {type:"integer"}
TILE_SIZE = 640 #@param {type:"integer"}
TILE_OVERLAP = 0.25 #@param {type:"number"}
REPORT_SCORE = 0.25 #@param {type:"number"}
SAVE_MIN_SCORE = 0.001 #@param {type:"number"}
MAX_DETECTIONS = 500 #@param {type:"integer"}
if not 0 < SHORT_EDGE <= LONG_EDGE or TILE_SIZE <= 0 or MAX_DETECTIONS <= 0:
    raise ValueError("해상도와 최대 탐지 개수는 양수로 설정하세요.")
if not 0 <= TILE_OVERLAP < 1:
    raise ValueError("겹침은 0 이상 1 미만이어야 합니다.")
SETTINGS = dict(repo=str(REPO), checkpoint=str(CHECKPOINT), mode=MODE,
    long_edge=LONG_EDGE, short_edge=SHORT_EDGE, tile_size=TILE_SIZE,
    tile_overlap=TILE_OVERLAP, merge_iou=0.5, report_score=REPORT_SCORE,
    save_min_score=SAVE_MIN_SCORE, max_detections=MAX_DETECTIONS,
    target_names=["person water"])
SETTINGS_PATH = WORK / "settings.json"
SETTINGS_PATH.write_text(json.dumps(SETTINGS), encoding="utf-8")
SINGLE_OUT = STORE / ("single_image_" + MODE)
checked([PYTHON, HELPERS / "engine.py", "--settings", SETTINGS_PATH,
         "--image", IMAGE_PATH, "--out", SINGLE_OUT], ENV)
# 엄격한 가중치 로드와 실제 추론을 통과한 파일만 보관한다.
if CHECKPOINT.resolve() != WEIGHT_STORE.resolve():
    shutil.copy2(CHECKPOINT, WEIGHT_STORE)
display(DisplayImage(filename=str(SINGLE_OUT / "prediction.jpg"), width=1200))
record = json.loads((SINGLE_OUT / "predictions.json").read_text())
visible = sum(b[4] >= REPORT_SCORE for b in record["boxes"])
print("실제 추론 완료. 표시 임계값 이상 박스:", visible)
print("탐지 0개도 실행 성공일 수 있습니다. 성능은 정답과 비교해야 판단할 수 있습니다.")
print("추론·분할 병합 시간: %.3f초 / 최대 할당 VRAM: %.1f MiB" %
      (record["seconds"], record["peak_vram_mib"]))


## 8. 선택 사항: 두 데이터셋의 COCO JSON ZIP 준비

**여기부터는 데이터셋을 확보한 뒤 실행합니다.** 지금은 `PREPARE_DATASET = False`여도 1~7단계 결과를 마지막 셀에서 다운로드할 수 있습니다.

- [Roboflow 프로젝트 모음](https://universe.roboflow.com/count-people-at-beaches)에서 **AnnotationPiriapolis v4** 또는 **YoloBeachTrain v2**를 선택하고 **COCO JSON 형식**으로 내보내세요. Roboflow 모델 API로 원격 추론하는 방식은 사용하지 않습니다. 데이터 다운로드에 유료 결제를 요구하면 진행하지 마세요.
- 두 데이터셋은 한 번에 하나씩 평가합니다. `DATASET_NAME`을 바꾸고 8~11단계를 반복하세요.
- ZIP에 `train/`, `valid/`, `test/`와 각 분할의 `_annotations.coco.json`·이미지가 함께 있어야 합니다. **`SPLIT=all`은 세 분할을 모두 합칩니다.** 분할 간 중복되는 ID와 파일명은 구분하고 이미지를 복사하지 않습니다. 누락된 분할이 있으면 중단합니다.
- `upload`는 로컬 ZIP, `path`는 Drive/Colab의 기존 ZIP입니다. 큰 ZIP은 Drive에 보관하고 `path`로 사용하는 편이 편리합니다. API 키는 필요하지 않습니다.
- 전체 평가에는 배포된 증강 이미지도 포함합니다. 향후 이 데이터로 추가 학습한 모델의 독립 테스트 성능을 보고할 때는 학습 분할을 분리하세요.


In [ ]:
PREPARE_DATASET = False #@param {type:"boolean"}
DATASET_NAME = "AnnotationPiriapolis_v4" #@param ["AnnotationPiriapolis_v4", "YoloBeachTrain_v2"]
ZIP_SOURCE = "upload" #@param ["upload", "path"]
EXISTING_ZIP_PATH = "" #@param {type:"string"}
SPLIT = "all" #@param ["all", "test", "valid", "train"]
ANNOTATION_OVERRIDE = "" #@param {type:"string"}
ANNOTATIONS = None
if PREPARE_DATASET:
    if ZIP_SOURCE == "upload":
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("데이터셋 ZIP 한 개만 선택하세요.")
        DATA_ZIP = WORK / (DATASET_NAME + ".zip")
        DATA_ZIP.write_bytes(next(iter(uploaded.values())))
    else:
        DATA_ZIP = Path(EXISTING_ZIP_PATH)
    if not DATA_ZIP.is_file():
        raise FileNotFoundError("데이터셋 ZIP 경로를 확인하세요.")
    DATA_ROOT = WORK / "datasets" / DATASET_NAME
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP) as archive:
        members = archive.infolist()
        total = sum(m.file_size for m in members)
        print("ZIP 압축 해제 크기: %.2f GiB" % (total / 1024**3))
        if total + 2 * 1024**3 > shutil.disk_usage("/content").free:
            raise RuntimeError("압축 해제와 결과 저장을 위한 임시 디스크 여유가 부족합니다.")
        base = DATA_ROOT.resolve()
        for member in members:
            target = (base / member.filename).resolve()
            if target != base and base not in target.parents:
                raise ValueError("ZIP에 폴더 밖을 가리키는 경로가 있습니다.")
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("ZIP의 심볼릭 링크는 지원하지 않습니다.")
        archive.extractall(DATA_ROOT)
    if ANNOTATION_OVERRIDE:
        if SPLIT == "all":
            raise ValueError("전체 분할 모드에서는 ANNOTATION_OVERRIDE를 비워두세요.")
        ANNOTATIONS = Path(ANNOTATION_OVERRIDE)
    elif SPLIT == "all":
        sources = []
        for part in ("train", "valid", "test"):
            candidates = [p for p in DATA_ROOT.rglob("_annotations.coco.json") if p.parent.name == part]
            if len(candidates) != 1:
                raise ValueError(part + " 분할의 정답 파일이 정확히 한 개 필요합니다: " + str(candidates))
            sources.append((part, str(candidates[0])))
        ANNOTATIONS = DATA_ROOT / "all_annotations.coco.json"
        merge_code = "from metrics import merge_splits; merge_splits(%r, %r)" % (sources, str(ANNOTATIONS))
        checked([PYTHON, "-c", merge_code], ENV)
    else:
        candidates = [p for p in DATA_ROOT.rglob("_annotations.coco.json") if p.parent.name == SPLIT]
        if len(candidates) != 1:
            raise ValueError("분할의 정답 파일이 한 개로 정해지지 않습니다. ANNOTATION_OVERRIDE를 지정하세요. 후보: " +
                             str([str(p) for p in DATA_ROOT.rglob("_annotations.coco.json")]))
        ANNOTATIONS = candidates[0]
    if not ANNOTATIONS.is_file():
        raise FileNotFoundError("정답 JSON 파일이 없습니다.")
    print("선택한 데이터셋:", DATASET_NAME, "/ 분할:", SPLIT, "/ 정답:", ANNOTATIONS)
    if SPLIT == "train":
        print("학습 분할을 선택했습니다. 이 결과를 최종 테스트 성능으로 보고하지 마세요.")
    if SPLIT == "all":
        print("train·valid·test 통합 완료. 9단계에서 전체 이미지·대상 수를 확인하세요.")
else:
    print("데이터셋 준비 생략. 단일 이미지 결과만 다운로드할 수 있습니다.")


## 9. 정답 라벨·이미지 크기·작은 객체 수 확인

기본 양성은 `person water` 한 종류입니다. `person_water` 표기도 같은 이름으로 정규화합니다. 실제 이름이 다르면 출력된 종류를 보고 수정하세요. `person sand`와 `umbrella`를 물속 사람으로 합치지 않습니다.

주 집계는 **`s=√(박스 가로×박스 세로)`의 `2≤s<20`, `2≤s<8` 구간**입니다. AnnotationPiriapolis v4의 세 분할 전체라면 이미지 **635장**, 물속 사람 **2,904개**, `2≤s<20` 정답 **2,823개**, `2≤s<8` 정답 **263개**가 나와야 합니다. 이전의 긴 변 기준 `d≤20`도 보조 지표로 유지하며, 두 정의는 서로 다릅니다.

두 크기는 모두 **현재 배포된 이미지 픽셀** 기준입니다. 원본을 축소한 데이터라면 원래 카메라 영상의 픽셀 크기와 같지 않을 수 있습니다. 배포 이미지를 임의로 640×640으로 다시 저장하지 마세요. 이미지 크기와 정답 좌표가 다르면 중단합니다.

해수면 ROI는 기본 적용하지 않습니다. SSPNet은 사람 1종 모델이므로 모래사장 사람도 검출할 수 있고, 물속 사람 평가에서 그 예측은 오탐입니다. 해수면 ROI를 도입한다면 정답 위치로 자동 생성하지 말고 사전에 정한 같은 ROI를 모든 모델에 적용하세요.


In [ ]:
TARGET_NAMES = ["person water"]
if PREPARE_DATASET:
    probe = """import json; from metrics import load_dataset
_, summary = load_dataset(%r, %r)
print(json.dumps(summary, ensure_ascii=False, indent=2))
""" % (str(ANNOTATIONS), TARGET_NAMES)
    checked([PYTHON, "-c", probe], ENV)
else:
    print("8단계에서 데이터셋을 먼저 준비하세요.")


## 10. 작은 표본 평가 → 전체 평가

**전체 635장 평가는 `RUN_EVALUATION=True`, `MAX_IMAGES=0`으로 실행합니다.** 표본만 확인하려면 50으로 바꿀 수 있지만 이 경우 전체 결과가 아닙니다. 한 장씩 처리하고, 이미지가 끝날 때마다 완료 기록을 저장합니다. Drive를 켰다면 새 세션에서도 코드·동일 가중치·동일 데이터를 준비한 뒤 완료된 이미지를 재사용합니다.

캐시는 코드·모델 가중치·원본 정답·추론 설정·각 이미지의 SHA256을 검사합니다. 해상도·분할·가중치가 바뀌면 다른 캐시를 사용합니다. 보고 임계값만 바꾸면 저장된 저점수 예측을 재사용합니다. 같은 저장 폴더에서 동시에 두 평가를 실행하지 마세요.

평가 방식은 이 셀에서 직접 고릅니다. 기본 `full`은 사진 전체를 검사합니다. `tiles`는 기본 320픽셀 패치이며 겹침을 지정할 수 있습니다. 외부 문서에 겹침·입력 해상도가 공개되지 않았으므로 해당 설정을 정확하게 재현했다고 표시하지 않습니다. 각 방식은 별도 폴더에 저장합니다.


In [ ]:
RUN_EVALUATION = False #@param {type:"boolean"}
MAX_IMAGES = 0 #@param {type:"integer"}
EVALUATION_MODE = "full" #@param ["full", "tiles"]
EVALUATION_TILE_SIZE = 320 #@param {type:"integer"}
EVALUATION_TILE_OVERLAP = 0.25 #@param {type:"number"}
MATCH_IOU = 0.5 #@param {type:"number"}
if RUN_EVALUATION:
    if not PREPARE_DATASET or ANNOTATIONS is None:
        raise RuntimeError("8~9단계에서 데이터셋과 정답을 먼저 확인하세요.")
    EVAL_SETTINGS = dict(SETTINGS, target_names=TARGET_NAMES,
                         dataset_name=DATASET_NAME, split=SPLIT, mode=EVALUATION_MODE,
                         tile_size=EVALUATION_TILE_SIZE, tile_overlap=EVALUATION_TILE_OVERLAP,
                         match_iou=MATCH_IOU)
    EVAL_SETTINGS_PATH = WORK / "evaluation_settings.json"
    EVAL_SETTINGS_PATH.write_text(json.dumps(EVAL_SETTINGS), encoding="utf-8")
    EVAL_LABEL = EVALUATION_MODE if EVALUATION_MODE == "full" else "tiles_%d_overlap%g" % (EVALUATION_TILE_SIZE,EVALUATION_TILE_OVERLAP)
    EVAL_OUT = STORE / "evaluation" / DATASET_NAME / SPLIT / EVAL_LABEL
    checked([PYTHON, HELPERS / "engine.py", "--settings", EVAL_SETTINGS_PATH,
             "--annotations", ANNOTATIONS, "--limit", MAX_IMAGES, "--out", EVAL_OUT], ENV)
else:
    print("평가 생략. 준비되면 RUN_EVALUATION을 True로 바꾸세요.")


## 11. 결과 읽는 방법

- **`문서용_결과요약.txt`**: 평가 범위·처리 방식·`2≤s<20` 및 `2≤s<8` 탐지 결과·정밀도·오탐·실제 임계값을 문서에 옮기기 쉬운 문장으로 저장합니다. 다른 모델의 결과나 비교표는 넣지 않습니다.
- **`geometric_2to20`, `geometric_2to8`**: `s=√(가로×세로)` 기준의 주 결과입니다. 각 항목의 `positives`는 정답 수, `hits`는 찾은 수, `recall`은 찾은 비율입니다.
- `metrics.json`·`metrics.csv`: 설정한 IoU와 점수 기준으로 전체 정밀도·재현율·오탐과 크기별 결과를 저장합니다. `by_split`에는 train·valid·test 각각의 결과도 남깁니다. 값은 0~1 비율이며 `null`은 분모가 없어 계산할 수 없다는 뜻입니다.
- `AP50_saved_predictions_101point`: 저장 점수 하한 이상·최대 탐지 개수 이내 예측에서 계산한 101점 보간 AP50입니다. 원 논문 TinyPerson의 별도 지표나 COCO mAP50:95와 혼동하지 마세요. AP50은 화면 표시 임계값과 별도로 저장된 전체 점수 범위를 사용합니다.
- 큰 물속 사람에 맞은 예측은 전체에서 정상 매칭한 뒤 작은 사람 재현율의 분모·분자에서 제외합니다. 중복 탐지와 사람이 없는 이미지의 예측은 오탐에 포함합니다.
- `predictions_coco.json`: 원본 이미지 좌표의 COCO 예측. `ground_truth_water.json`은 물속 사람만 남긴 정답이며 종류 ID는 1입니다. 다른 모델과 공통 평가기를 쓰는 데 활용할 수 있습니다.
- `run_manifest.json`·`resolved_config.py`·`installed_packages.txt`: 코드 버전, 가중치 해시, 실제 GPU, 이미지 ID, 해상도·분할·점수·탐지 개수 설정을 기록합니다.
- 처리 시간은 준비 호출 후의 추론·패치 병합 시간을 측정하고 이미지 읽기·파일 저장 시간은 제외합니다. GPU가 다르면 모델 속도를 직접 비교할 수 없습니다. 캐시를 재사용한 결과의 시간은 원래 추론 당시 측정값입니다.

문서에는 전체 분할 635장 평가인지와 `s=√(가로×세로)` 정의를 명시하세요. `s<20`은 가로·세로 모두 20 이하와 다릅니다. `tiny_le20`은 기존 긴 변 기준의 보조 지표입니다. 전체 분할 평가와 독립 test 분할 평가는 구분해 기록하세요.


In [ ]:
if RUN_EVALUATION:
    result = json.loads((EVAL_OUT / "metrics.json").read_text(encoding="utf-8"))
    display(result)
    print("주 결과 2≤√(가로×세로)<20:", result["geometric_2to20"])
    print("가장 작은 구간 2≤√(가로×세로)<8:", result["geometric_2to8"])
    print((EVAL_OUT / "문서용_결과요약.txt").read_text(encoding="utf-8"))
    if result["partial"]:
        print("현재는 표본 평가입니다. 전체 평가에는 MAX_IMAGES=0을 사용하세요.")


## 12. 결과 ZIP 다운로드

기본적으로 모델 가중치·원본 이미지·설치 환경은 ZIP에 넣지 않습니다. 결과 이미지, 예측, 평가표, 재시작 캐시, 설정을 포함합니다. Drive를 사용하지 않았다면 무료 런타임이 종료되기 전에 이 셀로 내려받으세요. 파일 다운로드는 브라우저에서 허용해야 할 수 있습니다.

중단된 평가를 Drive 없이 복구할 때는 내려받은 ZIP의 `evaluation/` 폴더를 새 세션의 `STORE` 아래에 복원하고, 같은 데이터·가중치·설정을 다시 준비하세요.


In [ ]:
DOWNLOAD_RESULTS = True #@param {type:"boolean"}
RESULT_ZIP = WORK / "SSPNet_results.zip"
with zipfile.ZipFile(RESULT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(STORE.rglob("*")):
        if path.is_file() and path.suffix.lower() not in (".pth", ".pt", ".tmp"):
            archive.write(path, path.relative_to(STORE))
print("결과 ZIP:", RESULT_ZIP, "%.2f MiB" % (RESULT_ZIP.stat().st_size / 1024**2))
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(RESULT_ZIP))


## 오류가 나면

| 증상 | 확인·대응 |
|---|---|
| GPU가 없음 | 무료 GPU가 배정될 때 다시 연결. 유료 업그레이드가 필수인 노트북이 아님 |
| Google Drive 다운로드 제한 | 5단계의 직접 업로드 또는 기존 가중치 경로 사용 |
| `No module named mmcv` | 기본 Python에서 실행했는지 확인. 모든 모델 실행은 4단계에서 만든 `PYTHON` 사용 |
| CUDA 연산·아키텍처 오류 | 4단계 GPU 검사 출력 확인. 다른 무료 런타임에서 재시도할 수 있으나 성공 보장은 없음 |
| 가중치 파라미터 불일치 | Faster R-CNN SSPNet용 가중치인지 확인. 오류를 무시해 임의 가중치로 평가하지 않음 |
| CUDA 메모리 부족 | 단일 이미지·낮은 해상도부터 확인. 변경된 해상도는 별도 실험으로 기록 |
| 물속 사람 라벨 없음 | 9단계의 실제 종류 이름을 보고 TARGET_NAMES 수정. 육상 사람을 임의로 합치지 않음 |
| test 분할 없음 | 검증 분할 사용 여부를 팀과 정하고 명시. train으로 자동 대체하지 않음 |
| 세션 중단 | 재설치 후 같은 설정으로 재실행. Drive에 저장된 완료 이미지부터 이어감 |

설치 참고: [SSPNet](https://github.com/MingboHong/SSPNet), [PyTorch 구버전](https://pytorch.org/get-started/previous-versions/), [CUDA 11.1·PyTorch 1.10용 MMCV wheel](https://download.openmmlab.com/mmcv/dist/cu111/torch1.10.0/index.html), [uv Python 설치](https://docs.astral.sh/uv/guides/install-python/), [Colab 사용 제한](https://research.google.com/colaboratory/faq.html).
